# 06.07 - PyTorch Tensors and a Tiny Classifier

**Daily output:** a tiny PyTorch classifier trained and evaluated correctly on toy data.

Today covers tensors, device, autograd, `nn.Module`, loss, optimizer, `model.train()`, `model.eval()`, and `torch.no_grad()`.

**Notebook type:** Practice notebook with theory, exercises, and TODO cells.


## Training Loop Contract

A PyTorch training step follows the same rhythm for toy data, CNNs, and transformers:

1. Move input and labels to the model device.
2. Run the forward pass.
3. Compute loss.
4. Clear old gradients with `optimizer.zero_grad()`.
5. Backpropagate with `loss.backward()`.
6. Update weights with `optimizer.step()`.

For `CrossEntropyLoss`, logits are `[batch, classes]` float values, and labels are `[batch]` `torch.long` class IDs.


In [21]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader, TensorDataset, random_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


device: cpu
torch: 2.13.0+cpu


## Tensor Basics

Inspect `.shape`, `.dtype`, `.device`, and `.requires_grad`. Most PyTorch errors are shape, dtype, or device errors.


In [4]:
# TODO 06-A: Tensor basics.
# Create:
# - x: shape [2, 3], dtype float32
# - y: shape [2], dtype long
# Print shape, dtype, and device.
# Move both tensors to `device`.
x = torch.ones((2,3),dtype = torch.float32)
y = torch.ones(2,dtype = torch.long)

## Autograd

If a tensor has `requires_grad=True`, PyTorch records operations on it. Calling `.backward()` computes gradients for leaf tensors.


In [9]:
# TODO 06-B: Autograd.
# Create w and b with requires_grad=True.
# Let pred = w * 3 + b and loss = (pred - 10) ** 2.
# Call backward and print gradients.
w = torch.tensor(2.0, requires_grad = True)
b = torch.tensor(1.0, requires_grad = True)
pred = w * 3 + b
loss = (pred - 10) ** 2
loss.backward()
print(w.grad)
print(b.grad)

tensor(-18.)
tensor(-6.)


## Toy 3-Class Data

This 2D dataset lets us train a classifier quickly while using the same code structure as a real image model.


In [19]:
# TODO 06-C: Toy 3-class data.
# Implement make_blobs(n_per_class, noise, seed).
# Return shuffled X with shape [N, 2] and y with shape [N].
# Then create train_ds, val_ds, train_loader, and val_loader.

def make_blobs(n_per_class=120, noise=0.65, seed=42):
    torch.manual_seed(seed)
    centers = torch.tensor([
        [-2.0,1.0],
        [1.0,2.0],
        [1.0,3.0],
    ])
    X_list = []
    y_list = []

    for idx, center in enumerate(centers) : 
        X_class = center + noise * torch.randn(n_per_class,2)
        y_class = torch.full((n_per_class,),idx,dtype = torch.long)
        X_list.append(X_class)
        y_list.append(y_class)
    
    X_torch = torch.cat(X_list,dim = 0)
    y_torch = torch.cat(y_list,dim = 0)
    perm = torch.randperm(X_torch.shape[0])
    X = X_torch[perm]
    y = y_torch[perm]
    return X,y

X,y = make_blobs()

print(X.shape)
print(y.shape)

dataset = TensorDataset(X,y)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset,[train_size,test_size])

train_loader = DataLoader(train_ds, batch_size = 32, shuffle = True)
val_loader = DataLoader(val_ds, batch_size = 32, shuffle = False)
# TODO: create X, y, TensorDataset, and DataLoader objects.


torch.Size([360, 2])
torch.Size([360])


## Model, Loss, Optimizer

`nn.Module` holds trainable layers. A linear classifier learns one score function per class. `CrossEntropyLoss` expects raw logits, not softmax probabilities.


In [22]:
# TODO 06-D: Model, loss, optimizer.
# Implement TinyLinearClassifier with one nn.Linear(2, 3).
# Create model, CrossEntropyLoss, and SGD optimizer.
# Run one batch through the model and print logits shape/loss.

class TinyLinearClassifier(nn.Module):
    def __init__(self, in_features=2, num_classes=3):
        super().__init__()
        self.linear = nn.Linear(in_features,num_classes)

    def forward(self, x):
        return self.linear(x)

model = TinyLinearClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = 0.1)
xb, yb = next(iter(train_loader))
logits = model(xb)
loss = criterion(logits,yb)

print("xb shape:", xb.shape)
print("yb shape:", yb.shape)
print("logits shape:", logits.shape)
print("loss:", loss.item())
# TODO: instantiate model, criterion, optimizer, and inspect one forward pass.


xb shape: torch.Size([32, 2])
yb shape: torch.Size([32])
logits shape: torch.Size([32, 3])
loss: 1.9990482330322266


## Train and Evaluate Correctly

`model.train()` enables training behavior. `model.eval()` switches to inference behavior. Use `torch.no_grad()` during validation so PyTorch does not store a gradient graph.


In [ ]:
# TODO 06-E: Train/evaluate functions.
# Implement:
# - train_one_epoch(model, loader, criterion, optimizer, device)
# - evaluate(model, loader, criterion, device)
#
# Remember train/eval mode, zero_grad, backward, step, no_grad.

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for xb, yb in loader : 
        xb = xb.to(device)
        yb = yb.to(device)

        # Clear grads
        optimizer.zero_grad()
        
        logits = model(xb)
        loss = criterion(logits,yb)

        # Backward pass
        loss.backward()
        # Update paras
        optimizer.step()

        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size

        preds = logits.argmax(dim = 1)
        total_correct += (preds == yb).sum().item()
        total_samples += batch_size
    
    avg_loss = total_loss/total_samples
    accuracy = total_correct/total_samples
    return avg_loss, accuracy


def evaluate(model, loader, criterion, device):
    model.train()

    all_preds = []
    all_labels = []

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for xb, yb in loader : 
        xb = xb.to(device)
        yb = yb.to(device)

        # Clear grads        
        logits = model(xb)
        loss = criterion(logits,yb)

        # Update paras
        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size

        preds = logits.argmax(dim = 1)
        total_correct += (preds == yb).sum().item()
        total_samples += batch_size

        all_preds.append(preds)
        all_labels.append(yb)

    all_preds = torch.cat(all_preds, dim = 0)
    all_labels = torch.cat(all_labels, dim = 0)
    
    avg_loss = total_loss/total_samples
    accuracy = total_correct/total_samples
    return {"loss" : avg_loss, "accuracy" : accuracy, "preds" : all_preds, "labels" : all_labels}


In [38]:
# TODO 06-F: Training loop.
# Train for 30 epochs and print train/validation loss and accuracy.

EPOCH = 30
for i in range(EPOCH) : 
    train_one_epoch(model,train_loader,criterion,optimizer,device)

eva_train = evaluate(model,train_loader,criterion,device)
eva_val = evaluate(model,val_loader,criterion,device)
avg_train_loss, train_accuracy, train_preds, train_labels = eva_train["loss"], eva_train["accuracy"], eva_train["preds"], eva_train["labels"]
avg_val_loss, val_accuracy, val_preds, val_labels = eva_val["loss"], eva_val["accuracy"], eva_val["preds"], eva_val["labels"]

print("Average train loss:", avg_train_loss)
print("Train Accuracy:", train_accuracy)
print("Average val loss",avg_val_loss)
print("Val Accuracy:",val_accuracy)

Average train loss: 0.3349244362778134
Train Accuracy: 0.8472222222222222
Average val loss 0.323593666156133
Val Accuracy: 0.8888888888888888


## Macro-F1

Macro-F1 averages F1 across classes, so each class matters equally. This is more informative than accuracy when classes are imbalanced.


In [39]:
# TODO 06-G: Macro-F1.
# Implement per_class_f1(preds, labels, num_classes).
# Return class, precision, recall, f1, support for each class.
# Then compute macro-F1 on validation predictions.

def per_class_f1(preds, labels, num_classes):
    TP = torch.tensor([0]*num_classes)
    FP = torch.tensor([0]*num_classes)
    FN = torch.tensor([0]*num_classes)
    res = [{} for _ in range(num_classes)]
    macro_f1 = 0 
    for idx in range(num_classes) : 
        TP[idx] = ((preds == idx) & (labels == idx)).sum().item()
        FP[idx] = ((preds == idx) & (labels != idx)).sum().item()
        FN[idx] = ((preds != idx) & (labels == idx)).sum().item()
        precision = TP[idx]/(TP[idx] + FP[idx])
        recall = TP[idx]/(TP[idx] + FN[idx])
        f1 = 2 * precision * recall / (precision + recall)
        support = (labels == idx).sum().item()
        res[idx]["class"] = idx
        res[idx]["precision"] = precision
        res[idx]["recall"] = recall
        res[idx]["f1"] = f1
        res[idx]["support"] = support
    return res

res = per_class_f1(val_preds,val_labels,num_classes)
macro_f1 = 0
for idx in range(num_classes) : 
    print("Class",idx,"F1:",res[idx]["f1"].item())
    macro_f1 += res[idx]["f1"]
macro_f1 /= num_classes
print("Macro F1: ",macro_f1.item())
# TODO: evaluate model and print per-class F1 plus macro-F1.


Class 0 F1: 1.0
Class 1 F1: 0.7777777910232544
Class 2 F1: 0.875
Macro F1:  0.8842592239379883


## Common PyTorch Bugs

- Labels for `CrossEntropyLoss` must be `torch.long`.
- Logits for multi-class classification must be `[batch, classes]`.
- Model, inputs, and labels must be on the same device.
- Gradients accumulate unless you call `optimizer.zero_grad()`.
- Validation should use `model.eval()` and `torch.no_grad()`.


In [ ]:
# Debug examples: uncomment one at a time and read the error.
# criterion(model(xb.to(device)), yb.float().to(device))
# model.to(device)(xb)  # fails if device is cuda and xb stays on CPU


## Day 06 Checklist

Check input shape, label dtype, logits shape, device consistency, `model.train()` during training, `model.eval()` plus `torch.no_grad()` during validation, gradient clearing, and validation metrics computed on validation data.


## Test Cases

Run this cell after completing the TODO cells above. A correct implementation should print `Day 06 tests passed`.


In [40]:
def run_day06_tests():
    required_names = [
        "make_blobs",
        "TinyLinearClassifier",
        "train_one_epoch",
        "evaluate",
        "per_class_f1",
    ]
    for name in required_names:
        assert name in globals(), f"Missing function or class: {name}"
        assert callable(globals()[name]), f"{name} must be callable"

    X_test, y_test = make_blobs(n_per_class=8, noise=0.5, seed=123)
    assert X_test.shape == (24, 2), f"Expected X shape (24, 2), got {X_test.shape}"
    assert y_test.shape == (24,), f"Expected y shape (24,), got {y_test.shape}"
    assert X_test.dtype == torch.float32
    assert y_test.dtype == torch.long
    assert set(y_test.tolist()) == {0, 1, 2}

    test_ds = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_ds, batch_size=12, shuffle=False)
    test_model = TinyLinearClassifier().to(device)
    test_criterion = nn.CrossEntropyLoss()
    test_optimizer = torch.optim.SGD(test_model.parameters(), lr=0.1)

    xb, yb = next(iter(test_loader))
    logits = test_model(xb.to(device))
    assert logits.shape == (12, 3), f"Expected logits shape (12, 3), got {logits.shape}"

    train_loss, train_acc = train_one_epoch(test_model, test_loader, test_criterion, test_optimizer, device)
    assert isinstance(train_loss, float)
    assert 0.0 <= train_acc <= 1.0

    metrics = evaluate(test_model, test_loader, test_criterion, device)
    for key in ["loss", "accuracy", "preds", "labels"]:
        assert key in metrics, f"evaluate output missing key: {key}"
    assert len(metrics["preds"]) == len(y_test)
    assert len(metrics["labels"]) == len(y_test)
    assert 0.0 <= metrics["accuracy"] <= 1.0

    perfect_rows = per_class_f1(
        preds=torch.tensor([0, 1, 2, 0, 1, 2]),
        labels=torch.tensor([0, 1, 2, 0, 1, 2]),
        num_classes=3,
    )
    assert len(perfect_rows) == 3
    assert all(abs(row["f1"] - 1.0) < 1e-8 for row in perfect_rows)

    print("Day 06 tests passed")

run_day06_tests()


Day 06 tests passed
